# Data compilation script
Produces aggregated results over multiple training seeds.

Produces: `all_channels_results_qc_clean.csv`

In [1]:
import pandas as pd
from scipy.stats import pearsonr
import numpy as np
from functools import reduce
from scipy.stats import combine_pvalues
from statsmodels.stats.multitest import multipletests
import scanpy as sc
import anndata
from src.eval import calculate_results_per_batch
import torch

In [2]:
# Parameters
## Inputs
whitelist_df = pd.read_csv(f'../input_files/cid_whitelist.csv')
toremove_df = pd.read_csv(f'../input_files/cid_nuc_toremove.csv')
whitelist_ids = np.array(whitelist_df.columns)
remove_cell_ids = np.array(toremove_df.columns)
seeds = [0, 1, 2, 3, 4]
model_specs = { # Will be looked for in "out_dir/{model_path}/seed_X/test_preds.h5ad"
    "fucci405": {
        "path": f'fucci_final_405-random_resnet',
        "channel": "Hoechst",
    },
    "fucci_angle": {
        "path": f'fucci_final_488-561-seg-bg-random/mlp_fucci_angle_model',
        "channel": "FUCCI angle",
    },
    "fucci_seg": {
        "path": f'fucci_final_seg-bg-random_resnet',
        "channel": "Segmentation",
    },
}

## Outputs
out_dir = '../output_files'
fig_dir = f'{out_dir}/Figures'
output_single_qc = f'test_results_qc.csv' # Will be created for each seed, for each model in "out_dir/{model_path}/seed_X/"
output_aggregated_qc = f'{out_dir}/all_channels_results_qc_clean.csv'

#### Aggregate results

In [3]:
for spec in model_specs.values():
    print(spec['path'])
    for seed in seeds:
        path = out_dir + '/' + spec['path'] + '/seed_{}/test_preds.h5ad'
        test_adata = anndata.read_h5ad(path.format(seed))
        keep_idxs = np.intersect1d(test_adata.obs.index, whitelist_ids)
        test_adata_ = test_adata[keep_idxs]
        print(test_adata_.shape, test_adata.shape)
        results_df = calculate_results_per_batch(test_adata_, test_adata_.layers['pred'], test_adata_.X)
        results_df.to_csv(out_dir + '/' + spec['path'] + '/seed_{}/'.format(seed) + output_single_qc)

fucci_final_405-random_resnet
(1042, 2000) (1165, 2000)
(1049, 2000) (1165, 2000)
(1054, 2000) (1165, 2000)
(1055, 2000) (1165, 2000)
(1029, 2000) (1165, 2000)
fucci_final_488-561-seg-bg-random/mlp_fucci_angle_model
(1042, 2000) (1165, 2000)
(1049, 2000) (1165, 2000)
(1054, 2000) (1165, 2000)
(1055, 2000) (1165, 2000)
(1029, 2000) (1165, 2000)
fucci_final_seg-bg-random_resnet
(1042, 2000) (1165, 2000)
(1049, 2000) (1165, 2000)
(1054, 2000) (1165, 2000)
(1055, 2000) (1165, 2000)
(1029, 2000) (1165, 2000)


In [4]:
def calculate_consensus(path_template, seeds, alpha=0.05, metric='avg_Pearson'):
    results_dfs = []
    pvals_dfs = []
    for seed in seeds:
        # print(path_template.format(seed=seed))
        results_df = pd.read_csv(path_template.format(seed), index_col='genes')
        results_df[f'{metric}_{seed}'] = results_df[metric]

        # Combine p-values across batches
        test_batches = [s.split('_')[1] for s in results_df.columns if s.startswith('Pearson_')]
        batch_pval_columns = [f'pvals_{s}' for s in test_batches]
        pvalues = []
        for g, row in results_df[batch_pval_columns].iterrows():
            combined_pval = combine_pvalues(row.values, method='fisher')[1]
            pvalues.append(combined_pval)
        results_df[f'{metric}_pval'] = pvalues
        pvals_dfs.append(results_df[f'{metric}_pval'])
        results_dfs.append(results_df[f'{metric}_{seed}'])
    
    # Average results
    merged_df = reduce(lambda left, right: pd.merge(left, right, left_index=True, right_index=True), results_dfs)
    merged_df[metric] = merged_df.values.mean(axis=-1)

    # Combine p-values across seeds
    genes = merged_df.index
    combined_pvals_df = pd.DataFrame(index=genes, columns=['combined_pval'])
    for g in genes:
        pvals = [df.loc[g] for df in pvals_dfs]
        combined_pvals_df.loc[g] = combine_pvalues(pvals, method='fisher')[1]
    merged_df[f'{metric}_pval'] = combined_pvals_df[f'combined_pval']
    merged_df[f'{metric}_fdr'] =  multipletests(combined_pvals_df[f'combined_pval'], alpha=alpha, method='fdr_bh')[1]
    # false_discovery_control(combined_pvals_df[f'combined_pval'])
    return merged_df

In [5]:
dfs = []
for spec in model_specs.values():
    full_path = f"{out_dir}/{spec['path']}/seed_{{}}/{output_single_qc}"
    df = calculate_consensus(full_path, seeds, metric="avg_Pearson")
    df["channel"] = spec["channel"]
    dfs.append(df)

all_merged_df = pd.concat(dfs, axis=0)
all_merged_df = all_merged_df[['avg_Pearson', 'avg_Pearson_pval', 'avg_Pearson_fdr', 'channel']]
print(all_merged_df)

         avg_Pearson avg_Pearson_pval avg_Pearson_fdr       channel
genes                                                              
Top2a       0.405941              0.0             0.0       Hoechst
Prc1        0.390047              0.0             0.0       Hoechst
Nusap1      0.311610              0.0             0.0       Hoechst
Kif20b      0.274476              0.0             0.0       Hoechst
Spc25       0.310577              0.0             0.0       Hoechst
...              ...              ...             ...           ...
Gm11033    -0.011211         0.076219        0.166917  Segmentation
Mtfmt      -0.032991          0.00005        0.000226  Segmentation
Gm17114    -0.005255          0.00098        0.003659  Segmentation
Kyat3      -0.029009         0.411247        0.582688  Segmentation
Cdh12      -0.013452         0.546159        0.706764  Segmentation

[4185 rows x 4 columns]


In [6]:
df_wide = all_merged_df.reset_index().pivot(index='genes', columns='channel', values=['avg_Pearson', 'avg_Pearson_pval', 'avg_Pearson_fdr'])

gene_mask = df_wide['avg_Pearson'].isna().any(axis=1)
df_wide = df_wide.loc[~gene_mask]
df_wide.to_csv(output_aggregated_qc)
df_wide

avg_Pearson                        avg_Pearson_pval            \
channel       FUCCI angle   Hoechst Segmentation      FUCCI angle   Hoechst   
genes                                                                         
1110032A03Rik   -0.006698 -0.013374     0.017845         0.639998  0.384147   
1190007I07Rik    0.029123   0.02429     0.028198         0.490942  0.199073   
1500011K16Rik    0.036958  0.091253     0.073876         0.189336  0.000066   
1600010M07Rik    0.068109   0.06257     0.085401         0.000032   0.00693   
1600019K03Rik    0.086191  0.166697     0.186113         0.001039       0.0   
...                   ...       ...          ...              ...       ...   
mt-Atp8          0.093384  0.045537       0.1264              0.0  0.000016   
mt-Nd6           0.097788  0.057281     0.114128         0.000015  0.000004   
mt-Rnr1          0.136917  0.050095     0.135463              0.0       0.0   
mt-Rnr2          0.185605  0.134612     0.207727              0.0       0.0   
mt-Tm            0.010152  0.023579     0.018409         0.729584  0.666582   

                           avg_Pearson_fdr                         
channel       Segmentation     FUCCI angle   Hoechst Segmentation  
genes                                                              
1110032A03Rik     0.356891        0.752891  0.637958     0.529642  
1190007I07Rik     0.942655        0.630873  0.428436     0.974076  
1500011K16Rik      0.02322        0.324334  0.000477     0.062292  
1600010M07Rik     0.000085        0.000141  0.032549     0.000367  
1600019K03Rik          0.0        0.003696       0.0          0.0  
...                    ...             ...       ...          ...  
mt-Atp8                0.0             0.0   0.00013          0.0  
mt-Nd6                 0.0        0.000068  0.000031          0.0  
mt-Rnr1                0.0             0.0       0.0          0.0  
mt-Rnr2                0.0             0.0       0.0          0.0  
mt-Tm             0.059607        0.814216  0.836225     0.134768  

[1395 rows x 9 columns]